Dataset link: https://bit.ly/moviesdataset

### Importing the dependencies

In [1]:
import numpy as np
import pandas as pd
import difflib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Data Collection and Pre-Processing

In [2]:
# loading the data from the csv file to a pandas dataframe
movies_data = pd.read_csv('movies.csv')

In [3]:
import sys
!{sys.executable} -m pip install jinja2

In [4]:
# printing the first 5 rows of the dataframe
movies_data.head()

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


In [5]:
!pip install summarytools
from summarytools import dfSummary
dfSummary(movies_data)

No,Variable,Stats / Values,Freqs / (% of Valid),Graph,Missing
1,index[int64],Mean (sd) : 2401.0 (1386.7)min < med < max:0.0 < 2401.0 < 4802.0IQR (CV) : 2401.0 (1.7),"4,803 distinct values","<img src = ""data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAKoAAABGCAYAAABc8A97AAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjEwLjEsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvc2/+5QAAAAlwSFlzAAAPYQAAD2EBqD+naQAAAb5JREFUeJzt101qwlAUQOH3ghVbRRFRF9AdOHQRXWwX4QocdNKRM0FCgqmlxabEuWgSsT1wvnGu9w6Of7EsyyD9d8lfHyBdo3PpgRjjYwihG5q/EX4azradp+6m3t12/qssy0OjUKtIJ5PJy3A4HNfdejweH4qieB4MBu9Jknzfc566m3r3LebzPE9jjK/nYr30idqtIl0ul4fRaPRZZ/Fmsxmv1+vpYrF4m8/naa2rW85Td1PvbjufZVlvtVqNd7td9c3dKNSTKtLZbPZRZ3maptVPhtDv92vPtp2n7qbefYv5EMJp/hz/TAnBUIVgqEIwVCEYqhAMVQiGKgRDFYKhCsFQhWCoQjBUIRiqEAxVCIYqBEMVgqEKwVCFYKhCMFQhGKoQDFUIhioEQxWCoQrBUIVgqEIwVCEYqhAMVQiGKgRDFYKhCsFQhWCoQjBUIRiqEAxVCIYqBEMVgqEKwVCFYKhCMFQhGKoQDFUIhioEQxWCoQrBUIVgqEIwVCEYqhAMVQiGKgRDFYKhCsFQhWCoQjBUIRiqEAxVCIYqhM41D2VZ1qv7wvv9/jRTFEVvu90+3XOeupt6d2g5f01fv5KWOeZTqJ76AAAAAElFTkSuQmCC"">",0(0.0%)
2,budget[int64],Mean (sd) : 29045039.9 (40722391.3)min < med < max:0.0 < 15000000.0 < 380000000.0IQR (CV) : 39210000.0 (0.7),436 distinct values,"<img src = ""data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAKoAAABGCAYAAABc8A97AAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjEwLjEsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvc2/+5QAAAAlwSFlzAAAPYQAAD2EBqD+naQAAAitJREFUeJzt2kFu2kAYhuH5gSAnIBCiMjs22WfBkkP0sNl2ieRb9AJIFtjCMQXRqaZSl1XDOC7+yPusmWEQr8aYsXnvHdB1vVsvAHiPwb9eYGaPzrmhi3Py3teRY4H3hRoinc/nXyeTycxFKMtyZ2avxIq2d9RhiHS9XtfT6fR4zcRFUSRZls3yPA+7MaGi3Ut/ECJN0/QtYv7wswFojJspSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSCBUSBi0Ofnlcnlwzk3MLHaKk/e+/thVQVFroVZV9XA+n1+Wy2W/3+8fY+Yoy3JnZq/EitZCPZ1OgyRJnlar1XGxWOyuHV8URZJl2SzP86FzjlA/uVYv/cFoNDqmafoWOfzxg5cDUdxMQQKhQgKhQgKhQgKhQgKhQkLrf0/d8GSLU6070tlQm55scap1XzobapOTrXCqtdls0jzPv5hZGbsEIu+Ozoba5GSL5wzuT+dDvdVzBg135HCT+tPFYSePDTV8ce5Kh8Ph95iqqpLtdvv0P8f/GRurruvoHTncAFZV9Twej7/3er3zte+93+8PZvbNORd1JXC6fnjv//qZfwFVF8yKAyrjxAAAAABJRU5ErkJggg=="">",0(0.0%)
3,genres[object],1. Drama2. Comedy3. Drama Romance4. Comedy Romance5. Comedy Drama6. Comedy Drama Romance7. Horror Thriller8. Documentary9. Horror10. Drama Thriller11. other,"370 (7.7%)282 (5.9%)164 (3.4%)144 (3.0%)142 (3.0%)109 (2.3%)88 (1.8%)68 (1.4%)64 (1.3%)62 (1.3%)3,310 (68.9%)","<img src = ""data:image/png;base64, iVBORw0KGgoAAAANSUhEUgAAAJsAAAD+CAYAAAAtWHdlAAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjEwLjEsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvc2/+5QAAAAlwSFlzAAAPYQAAD2EBqD+naQAABBtJREFUeJzt3VFqWmkAhmEN0jY3kRBwGVmAi5jFziJcQPYhg7mobW4sP7TDUDrEzOS8JzbPAw0UFIW+HOvh/9rl6XRaQOEqeRUQGyWxkREbGbGRERsZsZERGxmxkVkuFovrxWLx4QXPeTqdTscJ3xO/qdXd3d0fNzc3t+c+4fHx8a/lcvmn4Hip1Qhtu90e1+v1l+cefDgcPu12u9v9fj+uhGLjRVbjxwhts9l8PvM542MXXswXBDJiIyM2MmIjIzYyYiMjNjJiIyM2MmIjIzYyYiMjNjJioz1iNM6pnfPgcx8Hv7IaJ2/Hgchzz6mNx4+j4ec8Fv7JBoHM0j+ZxVu7srma0ayrLKpI1lUWVdTrKosq/jc3dcmIjYzYyIiNjNjIiI2M2MiIjYzYyIiNjNjIiI2M2Hg7gxcjF9LBi5ELr8GxcDIGL8x+ZXMloxm8GLiQDF4MXKgHLwYuvDo3dcmIjYzYyIiNjNjIiI2M2MiIjYzYyIiNjNjIiI2M2Jhv8GLgQjp4MXBhCo6FkzF4IeMLAhkfo2Ssq8hYV5GxriLjCwIZsZERGxmxkREbGbGRERsZsZERGxmxkREbGbGRERsZ6yoy1lVkHAsnY11FxpWNjMELGYMXMgYvZNzUJSM2MmIjIzYyYiMjNjJiIyM2MmIjIzYyYiMjNjJiI2PwQsbghYxj4WQMXpj1yuaqRjN4MXYhGbwYu1APXoxdmISbumTERkZsZMRGRmxkxEZGbGTERkZsZMRGRmxkxEZGbGTExjzrKssq0nWVZRVTsUEgY13FbFc2VzWadZVlFVO6+rGuGr++R/fzOh5ed131/feWVUzGTV0yYiMjNjJiIyM2MmIjIzYyYiMjNjJiIyM2MmIjIzbm+++EoBi8GLswKcfCyRi8MMuVzVWNZvBi7EIyeLm/v18Yu5DcZ7u+vv46+Svx7rmpS0ZsZMRGRmxkxEZGbGTERkZsZMRGRmxkxEZGbGTERhvb8Xj82L0k79XVODT58PBg7MLkHAsnY/BCxhcEMj5GyVhXkbGuImNdRcYXBDJiIyM2MmIjIzYyYiMjNjJiIyM2MmIjIzYyYiMjNjLWVWSsq8g4Fk7GuoqMKxsZgxcyBi9kDF7IuKlLRmxkxEZGbGTERkZsZMRGRmxkxEZGbGTERkZsZMRGxuCFjMELGcfCyRi8MMuVjbfv6ZI/ff4evMz9RnjepY+SViO07XZ7XK/XX+Z+M/y7w+Hwabfb3e73+/EpdJmxjR8jtM1m83nuN8Ozxl95LpabumTERkZsZMRGRmxkxEZGbGTERkZsZMRGRmxkxEZGbLSnPsbxle4l+S9+hz+j1TiQN85JXfr

In [6]:
movies_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 4803 non-null   int64  
 1   budget                4803 non-null   int64  
 2   genres                4775 non-null   object 
 3   homepage              1712 non-null   object 
 4   id                    4803 non-null   int64  
 5   keywords              4391 non-null   object 
 6   original_language     4803 non-null   object 
 7   original_title        4803 non-null   object 
 8   overview              4800 non-null   object 
 9   popularity            4803 non-null   float64
 10  production_companies  4803 non-null   object 
 11  production_countries  4803 non-null   object 
 12  release_date          4802 non-null   object 
 13  revenue               4803 non-null   int64  
 14  runtime               4801 non-null   float64
 15  spoken_languages     

In [7]:
# number of rows and columns in the data frame

movies_data.shape

(4803, 24)

In [8]:
# selecting the relevant features for recommendation

selected_features = ['genres','keywords','tagline','cast','director']
print(selected_features)

['genres', 'keywords', 'tagline', 'cast', 'director']


In [9]:
# replacing the null values with null string

for feature in selected_features:
  movies_data[feature] = movies_data[feature].fillna('')

In [10]:
# combining all the 5 selected features

combined_features = movies_data['genres']+' '+movies_data['keywords']+' '+movies_data['tagline']+' '+movies_data['cast']+' '+movies_data['director']

In [11]:
print(combined_features)

0       Action Adventure Fantasy Science Fiction cultu...
1       Adventure Fantasy Action ocean drug abuse exot...
2       Action Adventure Crime spy based on novel secr...
3       Action Crime Drama Thriller dc comics crime fi...
4       Action Adventure Science Fiction based on nove...
                              ...                        
4798    Action Crime Thriller united states\u2013mexic...
4799    Comedy Romance  A newlywed couple's honeymoon ...
4800    Comedy Drama Romance TV Movie date love at fir...
4801      A New Yorker in Shanghai Daniel Henney Eliza...
4802    Documentary obsession camcorder crush dream gi...
Length: 4803, dtype: object


In [12]:
# converting the text data to feature vectors

vectorizer = TfidfVectorizer()

In [13]:
feature_vectors = vectorizer.fit_transform(combined_features)

In [14]:
print(feature_vectors)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 124266 stored elements and shape (4803, 17318)>
  Coords	Values
  (0, 201)	0.07860022416510505
  (0, 274)	0.09021200873707368
  (0, 5274)	0.11108562744414445
  (0, 13599)	0.1036413987316636
  (0, 5437)	0.1036413987316636
  (0, 3678)	0.21392179219912877
  (0, 3065)	0.22208377802661425
  (0, 5836)	0.1646750903586285
  (0, 14378)	0.33962752210959823
  (0, 16587)	0.12549432354918996
  (0, 3225)	0.24960162956997736
  (0, 14271)	0.21392179219912877
  (0, 4945)	0.24025852494110758
  (0, 15261)	0.07095833561276566
  (0, 16998)	0.1282126322850579
  (0, 11192)	0.09049319826481456
  (0, 11503)	0.27211310056983656
  (0, 13349)	0.15021264094167086
  (0, 17007)	0.23643326319898797
  (0, 17290)	0.20197912553916567
  (0, 13319)	0.2177470539412484
  (0, 14064)	0.20596090415084142
  (0, 16668)	0.19843263965100372
  (0, 14608)	0.15150672398763912
  (0, 8756)	0.22709015857011816
  :	:
  (4801, 403)	0.17727585190343229
  (4801, 4835)	0.247137650

Cosine Similarity

In [15]:
# getting the similarity scores using cosine similarity
similarity = cosine_similarity(feature_vectors)

In [16]:
print(similarity)

[[1.         0.07219487 0.037733   ... 0.         0.         0.        ]
 [0.07219487 1.         0.03281499 ... 0.03575545 0.         0.        ]
 [0.037733   0.03281499 1.         ... 0.         0.05389661 0.        ]
 ...
 [0.         0.03575545 0.         ... 1.         0.         0.02651502]
 [0.         0.         0.05389661 ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.02651502 0.         1.        ]]


In [17]:
print(similarity.shape)

(4803, 4803)


Getting the movie name from the user

In [18]:
# getting the movie name from the user
movie_name = input('Enter your favourite movie name : ')

In [19]:
# creating a list with all the movie names given in the dataset
list_of_all_titles = movies_data['title'].tolist()
print(list_of_all_titles)

['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre', 'The Dark Knight Rises', 'John Carter', 'Spider-Man 3', 'Tangled', 'Avengers: Age of Ultron', 'Harry Potter and the Half-Blood Prince', 'Batman v Superman: Dawn of Justice', 'Superman Returns', 'Quantum of Solace', "Pirates of the Caribbean: Dead Man's Chest", 'The Lone Ranger', 'Man of Steel', 'The Chronicles of Narnia: Prince Caspian', 'The Avengers', 'Pirates of the Caribbean: On Stranger Tides', 'Men in Black 3', 'The Hobbit: The Battle of the Five Armies', 'The Amazing Spider-Man', 'Robin Hood', 'The Hobbit: The Desolation of Smaug', 'The Golden Compass', 'King Kong', 'Titanic', 'Captain America: Civil War', 'Battleship', 'Jurassic World', 'Skyfall', 'Spider-Man 2', 'Iron Man 3', 'Alice in Wonderland', 'X-Men: The Last Stand', 'Monsters University', 'Transformers: Revenge of the Fallen', 'Transformers: Age of Extinction', 'Oz: The Great and Powerful', 'The Amazing Spider-Man 2', 'TRON: Legacy', 'Cars 2', 'Green Lant

In [20]:
# finding the close match for the movie name given by the user
find_close_match = difflib.get_close_matches(movie_name, list_of_all_titles)
print(find_close_match)

['Nixon', 'Airborne', 'Prison']


In [21]:
close_match = find_close_match[0]
print(close_match)

Nixon


In [22]:
# finding the index of the movie with title
index_of_the_movie = movies_data[movies_data.title == close_match]['index'].values[0]
print(index_of_the_movie)

1091


In [23]:
# getting a list of similar movies

similarity_score = list(enumerate(similarity[index_of_the_movie]))
print(similarity_score)

[(0, np.float64(0.018519136073326208)), (1, np.float64(0.02374353974959803)), (2, np.float64(0.0)), (3, np.float64(0.007903748107679404)), (4, np.float64(0.08713410191090512)), (5, np.float64(0.004498320103775112)), (6, np.float64(0.0)), (7, np.float64(0.0)), (8, np.float64(0.0)), (9, np.float64(0.0)), (10, np.float64(0.02082976462842167)), (11, np.float64(0.0)), (12, np.float64(0.0)), (13, np.float64(0.0040030573893398355)), (14, np.float64(0.018398462783327824)), (15, np.float64(0.0)), (16, np.float64(0.0)), (17, np.float64(0.0)), (18, np.float64(0.06440567924308724)), (19, np.float64(0.007531338515282977)), (20, np.float64(0.03179588243158381)), (21, np.float64(0.0)), (22, np.float64(0.003921837068167465)), (23, np.float64(0.007509699456013058)), (24, np.float64(0.024148999389780556)), (25, np.float64(0.002754960986859542)), (26, np.float64(0.0422881823871899)), (27, np.float64(0.003955652055767714)), (28, np.float64(0.004342839151920344)), (29, np.float64(0.02435215041180489)), (30

In [24]:
len(similarity_score)

4803

In [25]:
# sorting the movies based on their similarity score

sorted_similar_movies = sorted(similarity_score, key = lambda x:x[1], reverse = True)
print(sorted_similar_movies)

[(1091, np.float64(1.0)), (656, np.float64(0.21917862344912128)), (1355, np.float64(0.20804891155595603)), (1814, np.float64(0.18256368138076728)), (2128, np.float64(0.16268044685774716)), (1687, np.float64(0.15808019654828176)), (2078, np.float64(0.1434688269895359)), (4152, np.float64(0.13842955694855835)), (401, np.float64(0.13049956805983598)), (1915, np.float64(0.11792887417890514)), (2629, np.float64(0.11752560161124763)), (2592, np.float64(0.11725312444960423)), (2763, np.float64(0.11446793815913281)), (2359, np.float64(0.11040874995836135)), (1836, np.float64(0.10957328736722789)), (2547, np.float64(0.10662047754392087)), (681, np.float64(0.10488318294542939)), (2168, np.float64(0.10461271223238493)), (4073, np.float64(0.10308755603980099)), (1921, np.float64(0.10273550761260425)), (2941, np.float64(0.1010430657238445)), (753, np.float64(0.09936392813237993)), (4110, np.float64(0.09845358638900818)), (3472, np.float64(0.09830889051083119)), (1351, np.float64(0.09812641249672396

In [26]:
# print the name of similar movies based on the index

print('Movies suggested for you : \n')

i = 1

for movie in sorted_similar_movies:
  index = movie[0]
  title_from_index = movies_data[movies_data.index==index]['title'].values[0]
  if (i<30):
    print(i, '.',title_from_index)
    i+=1

Movies suggested for you : 

1 . Nixon
2 . Primary Colors
3 . Head of State
4 . W.
5 . Man of the Year
6 . Frost/Nixon
7 . Swing Vote
8 . Straight A's
9 . Enemy at the Gates
10 . The Conspirator
11 . You Will Meet a Tall Dark Stranger
12 . Highlander: Endgame
13 . Dick
14 . Blow Out
15 . Tombstone
16 . The Theory of Everything
17 . The American President
18 . Appaloosa
19 . Sleeper
20 . Michael Collins
21 . Frailty
22 . The Sentinel
23 . Inside Deep Throat
24 . Pollock
25 . The Rite
26 . The Great Debaters
27 . Idiocracy
28 . Lion of the Desert
29 . Mr. Holland's Opus


Movie Recommendation Sytem

In [27]:
movie_name = input(' Enter your favourite movie name : ')

list_of_all_titles = movies_data['title'].tolist()

find_close_match = difflib.get_close_matches(movie_name, list_of_all_titles)

close_match = find_close_match[0]

index_of_the_movie = movies_data[movies_data.title == close_match]['index'].values[0]

similarity_score = list(enumerate(similarity[index_of_the_movie]))

sorted_similar_movies = sorted(similarity_score, key = lambda x:x[1], reverse = True)

print('Movies suggested for you : \n')

i = 1

for movie in sorted_similar_movies:
  index = movie[0]
  title_from_index = movies_data[movies_data.index==index]['title'].values[0]
  if (i<30):
    print(i, '.',title_from_index)
    i+=1

Movies suggested for you : 

1 . Hero
2 . 2046
3 . The Grandmaster
4 . House of Flying Daggers
5 . Ip Man 3
6 . Coming Home
7 . The Flowers of War
8 . Red Cliff
9 . Highlander: Endgame
10 . Bodyguards and Assassins
11 . Crouching Tiger, Hidden Dragon
12 . Curse of the Golden Flower
13 . Rush Hour 2
14 . Cold Mountain
15 . Memoirs of a Geisha
16 . A Woman, a Gun and a Noodle Shop
17 . My Lucky Star
18 . Tombstone
19 . A Civil Action
20 . The Last Legion
21 . Brokeback Mountain
22 . The Count of Monte Cristo
23 . The One
24 . The Forbidden Kingdom
25 . The Mummy: Tomb of the Dragon Emperor
26 . Faster
27 . Bambi
28 . The Warlords
29 . The Fall of the Roman Empire
